In [1]:
import pathlib

import numpy as np
import scipy.sparse as sp
import scipy.io
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import networkx as nx
import utils.preprocess
from sklearn.model_selection import train_test_split

save_prefix = r'./data/Aminer2_processed/graph_split/'
# save_prefix = r'D:\OneDrive\PhD\毕业\OHNN\demo_data\DBLP_processed/'
# read_perfix = 'demo_data/'
read_perfix = r'./data/Aminer/'
num_ntypes = 3

## load

In [2]:
node = pd.read_csv(read_perfix + 'node.dat', sep = '\t\t', header=None)
link = pd.read_csv(read_perfix + 'link.dat', sep = '\t', header=None)
meta = pd.read_csv(read_perfix + 'meta.dat', sep = '\t', header=None)
label = pd.read_csv(read_perfix + 'label.dat', sep = '\t', header=None)

d:\anaconda3\envs\ohnn\lib\site-packages\pandas\util\_decorators.py:311: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return func(*args, **kwargs)


In [3]:
meta

,0
0,Node Total: Count 55783
1,Node Type_0: Count 6564
2,Node Type_1: Count 13329
3,Node Type_2: Count 35890
4,Node type:
5,paper 0
6,author 1
7,reference 2
8,Relation:
9,source target edge_type


In [4]:
link.head(11)

,0,1,2,3
0,0,7590,0,1.0
1,7590,0,1,1.0
2,0,7591,0,1.0
3,7591,0,1,1.0
4,1,7592,0,1.0
5,7592,1,1,1.0
6,1,7594,0,1.0
7,7594,1,1,1.0
8,1,7595,0,1.0
9,7595,1,1,1.0


In [5]:
np.unique(link[2])

array([0, 1, 2, 3], dtype=int64)

## feature

In [6]:
(node[1]==0).values.nonzero()[0].shape

(6564,)

## label

In [7]:
label.shape

(1969, 4)

In [8]:
label.head(10)

,0,1,2,3
0,1357,NaN,0,1
1,4548,NaN,0,3
2,1033,NaN,0,0
3,2680,NaN,0,2
4,5116,NaN,0,3
5,1129,NaN,0,0
6,6355,NaN,0,3
7,999,NaN,0,0
8,4934,NaN,0,3
9,4150,NaN,0,3


In [9]:
[np.unique(label[p]) for p in label.columns]

[array([   2,    3,    8, ..., 6556, 6559, 6562], dtype=int64),
 array([nan]),
 array([0], dtype=int64),
 array([0, 1, 2, 3], dtype=int64)]

In [10]:
label[[0,3]]

,0,3
0,1357,1
1,4548,3
2,1033,0
3,2680,2
4,5116,3
...,...,...
1964,4611,3
1965,3080,3
1966,2916,2
1967,6372,3


In [11]:
# author2id = pd.read_csv(read_perfix + 'author2id2.txt', sep = '\t', header=None)
# conf2id = pd.read_csv(read_perfix + 'conf2id.txt', sep = '\t', header=None)
# label2id = pd.read_csv(read_perfix + 'label2id.txt', sep = '\t', header=None)
# paper_author = pd.read_csv(read_perfix + 'paper_author.txt', sep = '\t', header=None)
# paper_conference = pd.read_csv(read_perfix + 'paper_conference.txt', sep = '\t', header=None)
# paper_label = pd.read_csv(read_perfix + 'paper_label.txt', sep = '\t', header=None)
# paper_type = pd.read_csv(read_perfix + 'paper_type.txt', sep = '\t', header=None)
# paper_year = pd.read_csv(read_perfix + 'paper_year.txt', sep = '\t', header=None)


# data = [author2id, conf2id, label2id, paper_author, paper_conference, paper_label, paper_type, paper_year]
# [p.shape for p in data]

## type_mask

In [12]:
edge0 = link[link[2] == 0]
edge0.shape #(18007, 4)
edge1 = link[link[2] == 1]
edge1.shape #(18007, 4)
edge2 = link[link[2] == 2]
edge2.shape #(58831, 4)
edge3 = link[link[2] == 3]
edge3.shape #(58831, 4)

(58831, 4)

In [13]:
tmp = [max(link[link[2] == p][0]) for p in range(4)]
tmp

[6563, 19892, 6563, 55782]

In [14]:
tmp2 = [min(link[link[2] == p][0]) for p in range(4)]
tmp2

[0, 6564, 0, 19893]

In [15]:
tmp3 = [np.unique(link[link[2] == p][0]).shape for p in range(4)]
tmp3

[(6564,), (13329,), (6564,), (35890,)]

In [16]:
edge3.head(3)

,0,1,2,3
36015,20407,0,3,1.0
36017,21420,1,3,1.0
36019,21417,1,3,1.0


In [17]:
raw_nums = [6564, 13329, 35890]
nums = sum(raw_nums)
print(raw_nums)
print(nums)

prefix_operator = np.ones((num_ntypes+1, num_ntypes))
prefix_operator = np.tril(prefix_operator, k=-1)   # 下三角矩阵
prefix_operator = prefix_operator.dot(raw_nums).astype(int)
print(prefix_operator)

# 0 for movies 4661, 1 for directors 2270, 2 for actors 5841
type_mask = np.zeros(nums,dtype=int)
for i in range(num_ntypes):
    type_mask[prefix_operator[i]:prefix_operator[i+1]] = i

np.save(save_prefix + 'prefix_operator.npy',prefix_operator)
np.save(save_prefix + 'type_mask.npy',type_mask)

[6564, 13329, 35890]
55783
[    0  6564 19893 55783]


## adjM

In [18]:
adjM = sp.lil_matrix((nums,nums)) # 55783
for head,tail,_,__ in link.values:
    adjM[head,tail] = 1

scipy.sparse.save_npz(save_prefix + 'adjM.npz', adjM.tocsr())
# lil matrix cost 4s on 3700x platform

## edges for MHGCN

In [19]:
edges = [sp.lil_matrix((nums,nums)) for _ in range(4)] # 6 edges
for i in range(4):
    edge = link[link[2] == i]
    edges[i][edge[0],edge[1]] = 1

scipy.io.savemat(save_prefix + 'A.mat', {'edges': [edge.tocsr() for edge in edges]})
edges

[<55783x55783 sparse matrix of type '<class 'numpy.float64'>'
 	with 18007 stored elements in List of Lists format>,
 <55783x55783 sparse matrix of type '<class 'numpy.float64'>'
 	with 18007 stored elements in List of Lists format>,
 <55783x55783 sparse matrix of type '<class 'numpy.float64'>'
 	with 58831 stored elements in List of Lists format>,
 <55783x55783 sparse matrix of type '<class 'numpy.float64'>'
 	with 58831 stored elements in List of Lists format>]

## ontology

In [20]:
prefix_operator = np.load(save_prefix + 'prefix_operator.npy')
type_mask = np.load(save_prefix + 'type_mask.npy')
adjM = scipy.sparse.load_npz(save_prefix + 'adjM.npz')
ontology = {
    'stem':[1,0,2],
}
ontology_pairs = [[0,1],[0,2]]

In [21]:
link_intances = utils.preprocess.get_intances_by_pair(adjM, type_mask, ontology, prefix_operator)
#print(link_intances)
print('=======done=======')

# cost about 0s with sparse matrix csr 
# nodes 312776
# stem 165952

=======done=======


In [22]:
link_intances.keys()

dict_keys(['stem'])

In [23]:
link_intances['stem'].shape

(165988, 3)

In [24]:
ontology = {
    'stem':[1,0,2]
}
subgraphs = utils.preprocess.get_ontology_subgraphs_v3(ontology, link_intances)

subgraphs = subgraphs[subgraphs.columns.sort_values()]
print(len(subgraphs))
print(subgraphs)

# 0s
# 165952 subgraphs

165988
           0      1      2
0       2379   6564  26751
1       2379   6564  31183
2       2379   6564  31184
3       2379   6564  31322
4       2379   6564  31355
...      ...    ...    ...
165983  6559  19892  21064
165984  6559  19892  42366
165985  6559  19892  42369
165986  6559  19892  42372
165987  6559  19892  55759

[165988 rows x 3 columns]


In [25]:
prefix_operator = np.load(save_prefix + 'prefix_operator.npy')
type_mask = np.load(save_prefix + 'type_mask.npy')
adjM = scipy.sparse.load_npz(save_prefix + 'adjM.npz')
ontology = {
    'stem':[1,0,2],
}
ontology_pairs = [[0,1],[0,2]]
import time

In [26]:
t = time.time()
link_intances = utils.preprocess.get_intances_by_pair(adjM, type_mask, ontology, prefix_operator)
subgraphs = utils.preprocess.get_ontology_subgraphs_v3(ontology, link_intances)
print(time.time()-t)
subgraphs = subgraphs[subgraphs.columns.sort_values()]
print(len(subgraphs))
print(subgraphs)

0.07574701309204102
165988
           0      1      2
0       2379   6564  26751
1       2379   6564  31183
2       2379   6564  31184
3       2379   6564  31322
4       2379   6564  31355
...      ...    ...    ...
165983  6559  19892  21064
165984  6559  19892  42366
165985  6559  19892  42369
165986  6559  19892  42372
165987  6559  19892  55759

[165988 rows x 3 columns]


In [27]:
subgraphs.columns

Int64Index([0, 1, 2], dtype='int64')

## search incomplete, unfinished

In [71]:
ontology_pairs = [[0,1],[0,2]]
res_adj = utils.preprocess.find_res_adj2(adjM, subgraphs)
# 55s

In [72]:
incomplete_ontology_subgraphs, incomplete_subgraphs = utils.preprocess.find_incomplete_subgraph(adjM, type_mask, ontology_pairs, res_adj)
print(len(incomplete_ontology_subgraphs))
print(incomplete_subgraphs)
# 35min unfinished on 3700x

Sat Feb 24 14:04:40 2024, finding pairs...
Sat Feb 24 14:04:42 2024, finding pairs...
0
[]


## save

In [73]:
# create the directories if they do not exist
for i in ['complete','incomplete']:
    pathlib.Path(save_prefix + '{}'.format(i)).mkdir(parents=True, exist_ok=True)

# save data 

# mapping of node to subgraphs

# mapping of node to node pairs 

# save schema
np.save(save_prefix + 'complete/ontology.npy', ontology) # schema
np.save(save_prefix + 'ontology_pairs.npy', ontology_pairs)
# subgraph table
np.save(save_prefix + 'complete/subgraphs.npy', subgraphs)
# all nodes adjacency matrix
scipy.sparse.save_npz(save_prefix + 'adjM.npz', adjM)
# all nodes features one-hot
for i in range(num_ntypes):
    scipy.sparse.save_npz(save_prefix + 'features_{}.npz'.format(i), scipy.sparse.eye(raw_nums[i]).tocsr())
# all nodes (authors, papers, terms and conferences) type labels
np.save(save_prefix + 'node_types.npy', type_mask)
# type prefix
np.save(save_prefix + 'prefix_operator.npy', prefix_operator)
# paper labels
np.save(save_prefix + 'labels.npy', label[[0,3]]) # 4 class

### generate torch.sparse feature

In [29]:
adjM[np.unique(test_idx)].sum()/adjM.sum()

0.11373278846404122

In [30]:
adjM[np.unique(val_idx)].sum()/adjM.sum()

0.04492568781071866

## split

In [28]:
# subgraphs train/validation/test splits
rand_seed = 33333333
train_val_idx, test_idx = train_test_split(np.arange(adjM.shape[0]), test_size=0.115, random_state=rand_seed)
a = np.isin(subgraphs,test_idx)
a = a.sum(axis=1).astype('bool')
subgraphs_test = subgraphs[a]
subgraphs_tr_val = subgraphs[~a]
subgraphs[a].shape
print(subgraphs_test.shape[0]/len(subgraphs)) # 30% for test
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.05, random_state=rand_seed)
b = np.isin(subgraphs_tr_val,val_idx)
b = b.sum(axis=1).astype('bool')
subgraphs_val = subgraphs_tr_val[b]
subgraphs_train = subgraphs_tr_val[~b]
subgraphs_tr_val[b].shape
print(subgraphs_val.shape[0]/len(subgraphs)) # 10% for val
print(len(subgraphs_train)/len(subgraphs)) # 60% for train


0.304672626936887
0.10133262645492445
0.5939947466081885


In [ ]:

np.savez(save_prefix + 'complete/' + 'subgraphs_train_val_test.npz',
         subgraphs_train=subgraphs_train,
         subgraphs_val=subgraphs_val,
         subgraphs_test=subgraphs_test) # subgraph train&val&test
# node split
np.savez(save_prefix + 'complete/' + 'train_val_test_nodes.npz',
         train_nodes=train_idx,
         val_nodes=val_idx,
         test_nodes=test_idx) # nodes train&val&test

In [ ]:
mask = np.ones_like(adjM, dtype=bool)
for row in subgraphs.values:
    mask[np.ix_(row,row)] = False

In [ ]:
subgraphs.values[1]

In [ ]:
adjM[9]

In [ ]:
adjM[np.ix_([9,10,11,12,13,14,15], [79,80,81,82,83,84,85,86,87])]

In [ ]:
adjM[[9,10,11,12,13,14,15]][:,[79,80,81,82,83,84,85,86,87]]

In [ ]:
np.ix_([9,10,11,12,13,14,15], [79,80,81,82,83,84,85,86,87])

In [ ]:
((adjM[9] != 0))